# 어텐션이 포함된 Sequence-to-Sequence 작성하기

### 문제 설명
필요한 섹션을 완성하여 **어텐션이 포함된 시퀀스-투-시퀀스(Seq2Seq) 모델**을 구현합니다. 이 모델은 입력 시퀀스를 처리하는 **Encoder**와, 어텐션 메커니즘으로 출력 시퀀스를 생성하는 **Decoder**로 구성됩니다.

### 요구사항

1. **Encoder 클래스**:
   - **레이어**:
     - 입력 토큰을 밀집 벡터로 매핑하는 임베딩 레이어를 사용합니다.
     - 시퀀스의 시간적 의존성을 포착하기 위해 LSTM 레이어를 사용합니다.
   - **Forward Pass**:
     - 입력 시퀀스를 임베딩 레이어에 통과시킵니다.
     - 임베딩된 시퀀스를 LSTM에 전달합니다.
     - LSTM의 출력과 마지막 hidden state, cell state를 반환합니다.

2. **어텐션이 포함된 Decoder**:
   - **레이어**:
     - 출력 시퀀스 토큰을 처리하는 임베딩 레이어를 사용합니다.
     - Encoder 출력과 현재 Decoder hidden state 사이의 attention weight를 계산하는 어텐션 메커니즘을 구현합니다.
     - 어텐션에서 얻은 context vector와 현재 Decoder state를 사용해 다음 토큰을 예측하기 위해 LSTM 레이어를 사용합니다.
     - 다음 토큰을 예측하기 위한 완전 연결 출력 레이어를 사용합니다.
   - **Forward Pass**:
     - 입력을 임베딩 레이어에 통과시킵니다.
     - Decoder hidden state와 Encoder 출력을 사용해 attention weight를 계산합니다.
     - 어텐션 weight를 Encoder 출력에 적용해 context vector를 계산합니다.
     - context vector와 임베딩된 입력을 결합합니다.
     - 결합된 표현을 LSTM에 전달합니다.
     - LSTM 출력을 완전 연결 레이어에 통과시켜 다음 토큰을 예측합니다.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [33]:
#TODO: Implement the Encoder class
class Encoder(nn.Module):
    def __init__(self, input_dim, embed_dim, hidden_dim, num_layers):
        super(Encoder, self).__init__()
        # Add layers for embedding and LSTM
        self.embedding = nn.Embedding(input_dim, embed_dim)
        self.lstm = nn.LSTM(input_size = embed_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True)
    def forward(self, x):
        # Define the forward pass for the encoder
        x = self.embedding(x)
        x, (h, c) = self.lstm(x)
        return x, (h, c)



# Define the Decoder with Attention
#TODO: Implement the Decoder class with Attention
class Decoder(nn.Module):
    def __init__(self, output_dim, embed_dim, hidden_dim, num_layers, src_seq_length): # ?
        super(Decoder, self).__init__()
        # Add layers for embedding, attention, LSTM, and output
        self.embedding = nn.Embedding(output_dim, embed_dim)
        self.attention = nn.Linear(hidden_dim + embed_dim, src_seq_length)
        self.seq_len = src_seq_length
        self.attn_combine = nn.Linear(hidden_dim + embed_dim, embed_dim)
        self.lstm = nn.LSTM(input_size = embed_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, encoder_outputs, hidden, cell):
        # Define the forward pass for the decoder
        x = x.unsqueeze(1)  # Add sequence dimension
        embedded = self.embedding(x)
        # Attention mechanism
        concatenated_states = torch.cat((embedded.squeeze(1), hidden[-1]), dim=1)
        attention_weights = torch.softmax(self.attention(concatenated_states), dim=1)
        context_vector = torch.bmm(attention_weights.unsqueeze(1), encoder_outputs)

        # Combine context and embedded input
        combined = torch.cat((embedded.squeeze(1), context_vector.squeeze(1)), dim=1)
        combined = torch.tanh(self.attn_combine(combined)).unsqueeze(1)
        print(combined.shape)
        # LSTM and output
        lstm_out, (hidden, cell) = self.lstm(combined, (hidden, cell))
        output = self.fc_out(lstm_out.squeeze(1))
        return output, hidden, cell

In [34]:
# Define synthetic training data
torch.manual_seed(42)
src_vocab_size = 20
tgt_vocab_size = 20
src_seq_length = 10
tgt_seq_length = 12
batch_size = 16

src_data = torch.randint(0, src_vocab_size, (batch_size, src_seq_length))
tgt_data = torch.randint(0, tgt_vocab_size, (batch_size, tgt_seq_length))

# Initialize models, loss function, and optimizer
input_dim = src_vocab_size
output_dim = tgt_vocab_size
embed_dim = 32
hidden_dim = 64
num_layers = 2

encoder = Encoder(input_dim, embed_dim, hidden_dim, num_layers)
decoder = Decoder(output_dim, embed_dim, hidden_dim, num_layers, src_seq_length)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.001)

In [35]:
# Training loop
epochs = 100
for epoch in range(epochs):
    encoder_outputs, (hidden, cell) = encoder(src_data)
    loss = 0
    decoder_input = torch.zeros(batch_size, dtype=torch.long)  # Start token

    for t in range(tgt_seq_length):
        output, hidden, cell = decoder(decoder_input, encoder_outputs, hidden, cell)
        loss += criterion(output, tgt_data[:, t])
        decoder_input = tgt_data[:, t]  # Teacher forcing

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Log progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}] - Loss: {loss.item():.4f}")

torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 1, 32])
torch.Size([16, 

KeyboardInterrupt: 

In [19]:
# Test the sequence-to-sequence model with new input
test_input = torch.randint(0, src_vocab_size, (1, src_seq_length))
with torch.no_grad():
    encoder_outputs, (hidden, cell) = encoder(test_input)
    decoder_input = torch.zeros(1, dtype=torch.long)  # Start token
    output_sequence = []

    for _ in range(tgt_seq_length):
        output, hidden, cell = decoder(decoder_input, encoder_outputs, hidden, cell)
        predicted = output.argmax(1)
        output_sequence.append(predicted.item())
        decoder_input = predicted

    print(f"Input: {test_input.tolist()}, Output: {output_sequence}")

Input: [[3, 18, 4, 11, 8, 17, 12, 7, 18, 1]], Output: [13, 13, 2, 2, 2, 12, 12, 7, 7, 12, 12, 12]
